# BinaryTaskConfig: аргументы и YAML

Проверяются независимые способы конфигурации: Python-аргументы или основной YAML, а feature lists — непосредственно списками или абсолютными путями к отдельным YAML. Все четыре варианта реально обучаются с `hyperopt=True`, `n_trials=1`.

In [1]:
from pathlib import Path

import polars as pl
from IPython.display import display

from fmlib.automl import BinaryTask, BinaryTaskConfig

WORKSPACE = Path.cwd().parent.resolve()
TEST_DIR = Path.cwd() / 'examples/automl/tests'
TRAIN_PATH = WORKSPACE / 'data/binary/train'
VALID_PATH = WORKSPACE / 'data/binary/valid'
CAT_COLS = [f'cat_feature_{index}' for index in range(1, 6)]
NUM_COLS = [f'num_feature_{index}' for index in range(1, 6)]
CAT_YAML = (TEST_DIR / 'configs/features/categorical_columns.yaml').resolve()
NUM_YAML = (TEST_DIR / 'configs/features/numerical_columns.yaml').resolve()
SEARCH_SPACE = {'iterations': [16], 'depth': [2], 'l2_leaf_reg': [1.0], 'bagging_temperature': [0.25], 'learning_rate': [0.1]}

def python_config(categorical_columns, numerical_columns, name):
    return BinaryTaskConfig(environment={}, 
        env_type='local', backend='boosting', engine='catboost', device='gpu',
        target_column='target', client_id_column='epk_id', report_month_column='report_month',
        group_column='group', treatment_column='treatment', inverse_treatment=True, model_scope='product',
        categorical_columns=categorical_columns, numerical_columns=numerical_columns, hidden_state_columns=['seq_hidden_state'],
        hyperopt=True, optimization_metric='roc_auc', n_trials=1, random_state=42, verbose=False,
        search_space=SEARCH_SPACE,
        output_dir=WORKSPACE / f'outputs/notebook_tests/config_sources/{name}',
    )


In [2]:
configs = {
    'arguments + inline features': python_config(CAT_COLS, NUM_COLS, 'args_inline'),
    'arguments + feature YAML': python_config(CAT_YAML, NUM_YAML, 'args_feature_yaml'),
    'main YAML + feature YAML': BinaryTaskConfig.from_yaml(TEST_DIR / 'configs/binary_one_trial.yaml'),
    'main YAML + inline features': BinaryTaskConfig.from_yaml(TEST_DIR / 'configs/binary_inline_features.yaml'),
}

contract_fields = ('engine', 'target_column', 'client_id_column', 'report_month_column', 'group_column', 'treatment_column', 'inverse_treatment', 'model_scope', 'categorical_columns', 'numerical_columns', 'hidden_state_columns', 'hyperopt', 'n_trials')
baseline = configs['arguments + inline features']
for name, config in configs.items():
    assert all(getattr(config, field) == getattr(baseline, field) for field in contract_fields), name
    assert Path(config.categorical_columns[0]).name == config.categorical_columns[0]

display(pl.DataFrame({'source': list(configs), 'categorical_columns': [len(config.categorical_columns) for config in configs.values()], 'numerical_columns': [len(config.numerical_columns) for config in configs.values()], 'n_trials': [config.n_trials for config in configs.values()]}))


source,categorical_columns,numerical_columns,n_trials
str,i64,i64,i64
"""arguments + inline features""",5,5,1
"""arguments + feature YAML""",5,5,1
"""main YAML + feature YAML""",5,5,1
"""main YAML + inline features""",5,5,1


In [3]:
rows = []
for source, config in configs.items():
    task = BinaryTask(config)
    training = task.train(TRAIN_PATH, VALID_PATH)
    rows.append({'source': source, 'validation_roc_auc': training.validation_metrics['product'], 'best_params': str(training.best_params['product'])})

display(pl.DataFrame(rows))


2026-08-17 13:41:01,453 INFO run_id=a4049fbedcf04c5fbb64e5b32274b61c Starting action=train task_config=BinaryTaskConfig backend=boosting engine=catboost env_type=local


<workspace>\fmlib-main\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-08-17 13:41:03,466] A new study created in memory with name: no-name-21f3d865-2dc6-4d81-8d6a-fedd926ad5ec


[I 2026-08-17 13:41:07,008] Trial 0 finished with value: 0.7242174460395527 and parameters: {'iterations': 16, 'depth': 2, 'l2_leaf_reg': 1.0, 'bagging_temperature': 0.25, 'learning_rate': 0.1}. Best is trial 0 with value: 0.7242174460395527.


2026-08-17 13:41:07,012 INFO run_id=a4049fbedcf04c5fbb64e5b32274b61c Finished action=train


2026-08-17 13:41:07,018 INFO run_id=22e0a31ea54a479fa2a89e2661d3df49 Starting action=train task_config=BinaryTaskConfig backend=boosting engine=catboost env_type=local


...

2026-08-17 13:41:13,446 INFO run_id=7d8f63ef2c684688b25c8ff5a2ba7d4a Starting action=train task_config=BinaryTaskConfig backend=boosting engine=catboost env_type=local


[I 2026-08-17 13:41:14,830] A new study created in memory with name: no-name-6975af1e-f8a8-4e68-92b7-2dc12bb875e9


[I 2026-08-17 13:41:16,496] Trial 0 finished with value: 0.7242333449101337 and parameters: {'iterations': 16, 'depth': 2, 'l2_leaf_reg': 1.0, 'bagging_temperature': 0.25, 'learning_rate': 0.1}. Best is trial 0 with value: 0.7242333449101337.


2026-08-17 13:41:16,509 INFO run_id=7d8f63ef2c684688b25c8ff5a2ba7d4a Finished action=train


source,validation_roc_auc,best_params
str,f64,str
"""arguments + inline features""",0.724217,"""{'iterations': 16, 'depth': 2,…"
"""arguments + feature YAML""",0.724233,"""{'iterations': 16, 'depth': 2,…"
"""main YAML + feature YAML""",0.724217,"""{'iterations': 16, 'depth': 2,…"
"""main YAML + inline features""",0.724233,"""{'iterations': 16, 'depth': 2,…"


Первая таблица подтверждает одинаковый разрешённый контракт. Во второй показан результат реального обучения; при одинаковом seed и search space параметры и validation ROC AUC должны совпасть.